In [1]:
# Cell 1：安装依赖（已安装可跳过）
%pip install -q torch transformers scikit-learn plotly pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
from urllib.request import urlretrieve
import re
import pandas as pd
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
 
cilin_path = Path("cilin.txt")
cilin_url = (
    "https://raw.githubusercontent.com/"
    "HapuHXY/task2-cilin/master/new_cilin.txt"
)

if not cilin_path.exists():
    urlretrieve(cilin_url, cilin_path)

line_pattern = re.compile(r"^([A-L][a-z]\d{2}[A-Z]\d{2})([=#@])\s*(.*)$")
word_pattern = re.compile(r"^[\u4e00-\u9fff]{2,4}$")  # 保留 2~4 字中文词

rows = []
for line_id, line in enumerate(cilin_path.read_text(encoding="gb18030").splitlines()):
    match = line_pattern.match(line.strip())
    if not match:
        continue

    code, relation, words_text = match.groups()
    small_class = code[:4]
    words = words_text.split()

    for word in words:
        if word_pattern.fullmatch(word):
            rows.append({
                "word": word,
                "small_class": small_class,
                "cilin_code": code,
                "relation": relation,
                "line_id": line_id
            })

df = pd.DataFrame(rows)
df
    

,word,small_class,cilin_code,relation,line_id
0,人物,Aa01,Aa01A01,=,5758
1,人士,Aa01,Aa01A01,=,5758
2,人氏,Aa01,Aa01A01,=,5758
3,人选,Aa01,Aa01A01,=,5758
4,人类,Aa01,Aa01A02,=,5759
...,...,...,...,...,...
79794,好说,La06,La06D01,=,23572
79795,彼此彼此,La06,La06D01,=,23572
79796,岂敢,La06,La06D02,@,23573
79797,过奖,La06,La06D03,=,23574


In [4]:
title_map = (
    df.sort_values("line_id")
    .groupby("small_class", sort=False)["word"]
    .first()
)
df["small_label"] = df["small_class"] + " · " + df["small_class"].map(title_map)
n_classes_per_word = df.groupby("word")["small_class"].nunique()
df = df[df["word"].map(n_classes_per_word).eq(1)].copy()
df


,word,small_class,cilin_code,relation,line_id,small_label
0,人物,Aa01,Aa01A01,=,5758,Aa01 · 人物
1,人士,Aa01,Aa01A01,=,5758,Aa01 · 人物
2,人氏,Aa01,Aa01A01,=,5758,Aa01 · 人物
3,人选,Aa01,Aa01A01,=,5758,Aa01 · 人物
4,人类,Aa01,Aa01A02,=,5759,Aa01 · 人物
...,...,...,...,...,...,...
79792,不敢当,La06,La06D01,=,23572,La06 · 恭贺
79793,别客气,La06,La06D01,=,23572,La06 · 恭贺
79794,好说,La06,La06D01,=,23572,La06 · 恭贺
79795,彼此彼此,La06,La06D01,=,23572,La06 · 恭贺


In [5]:
N_CLASSES = 10
WORDS_PER_CLASS = 100

class_sizes = df.groupby("small_class").size()
eligible_classes = class_sizes[class_sizes >= WORDS_PER_CLASS].index.to_numpy()

chosen_classes = rng.choice(eligible_classes, size=N_CLASSES, replace=False)

samples = []
for small_class in chosen_classes:
    part = df[df["small_class"].eq(small_class)]
    samples.append(part.sample(n=WORDS_PER_CLASS, random_state=SEED))

sample_df = (
    pd.concat(samples, ignore_index=True)
    .sort_values(["small_class", "word"])
    .reset_index(drop=True)
)

print(sample_df.groupby("small_class").size())
print(f"\nTotal words: {len(sample_df)}")

small_class
Bd01    100
Be04    100
Bh13    100
Ca03    100
Ca25    100
Di18    100
Dk03    100
Dk19    100
Dn07    100
Ig01    100
dtype: int64

Total words: 1000


In [6]:
# save to local
sample_df.to_csv("cilin_1000_words.csv", index=False, encoding="utf-8-sig")

In [7]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-chinese"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

def encode_words(words, batch_size=32):
    all_vectors = []

    for start in range(0, len(words), batch_size):
        batch_words = words[start:start + batch_size]
        texts = [f"{word}" for word in batch_words]

        encoded = tokenizer(
            texts, 
            padding = True,
            truncation = True,
            return_tensors = "pt",
            return_offsets_mapping = True
        )

        offsets = encoded.pop("offset_mapping").numpy()
        model_inputs = {key: value.to(device) for key, value in encoded.items()}
        with torch.inference_mode():
            output = model(**model_inputs, output_hidden_states=True)
            hidden = torch.stack(output.hidden_states[-4:]).mean(dim=0)

        for i, word in enumerate(batch_words):

            target_start = 0
            target_end = target_start + len(word)
            starts = offsets[i, :, 0]
            ends = offsets[i, :, 1]
            target_mask = (starts >= target_start) & (ends <= target_end) & (ends > starts)

            vector = hidden[i, torch.tensor(target_mask, device=device)].mean(dim=0)
            all_vectors.append(vector.cpu().numpy())

    return np.stack(all_vectors, axis=0)

embeddings = encode_words(sample_df["word"].to_list(), batch_size=32)
print(f"Embeddings shape: {embeddings.shape}")

/opt/miniconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21497.62it/s]
[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loadin

Embeddings shape: (1000, 768)


In [10]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE


X = StandardScaler().fit_transform(embeddings)

X_pca = PCA(n_components=50, random_state=SEED).fit_transform(X)

coords_3d = TSNE(
    n_components = 3,
    perplexity = 35,
    learning_rate = "auto",
    init = "pca",
    max_iter = 1500,
    random_state = SEED
).fit_transform(X_pca)

plot_df = sample_df.copy()
plot_df[["x", "y", "z"]] = coords_3d
plot_df.head()

#save to local
plot_df.to_csv("cilin_1000_words_with_coords.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Cell 6：Plotly 交互式三维图
import plotly.express as px

fig = px.scatter_3d(
    plot_df,
    x="x",
    y="y",
    z="z",
    color="small_label",
    hover_name="word",
    hover_data={
        "small_class": True,
        "cilin_code": True,
        "x": False,
        "y": False,
        "z": False,
    },
    color_discrete_sequence=px.colors.qualitative.Dark24,
    title="Chinese BERT 词向量三维可视化：同义词词林小类",
)

fig.update_traces(
    marker=dict(size=4.5, opacity=0.80, line=dict(width=0.3, color="white"))
)

fig.update_layout(
    width=1000,
    height=800,
    legend_title_text="词林小类",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgb(248, 249, 252)",
    ),
    paper_bgcolor="white",
    margin=dict(l=0, r=0, t=60, b=0),
)

fig.show()
fig.write_html("cilin_bert_3d.html")

In [14]:
%pip install -U "nbformat>=4.2.0" ipython

Note: you may need to restart the kernel to use updated packages.


In [15]:
#plot the 3D scatter plot of the words colored by their small class labels using Plotly. The plot is interactive and allows for zooming, rotating, and hovering over points to see additional information about each word. The final plot is saved as an HTML file for easy sharing and viewing in a web browser.
import plotly.express as px

fig = px.scatter_3d(
    plot_df,
    x="x",
    y="y",
    z="z",
    color="small_label",
    hover_name="word",
    hover_data={
        "small_class": True,
        "cilin_code": True,
        "x": False,
        "y": False,
        "z": False,
    },
    color_discrete_sequence=px.colors.qualitative.Dark24,
    title="Chinese BERT 词向量三维可视化：同义词词林小类",
)

fig.update_traces(
    marker=dict(size=4.5, opacity=0.80, line=dict(width=0.3, color="white"))
)

fig.update_layout(
    width=1000,
    height=800,
    legend_title_text="词林小类",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgb(248, 249, 252)",
    ),
    paper_bgcolor="white",
    margin=dict(l=0, r=0, t=60, b=0),
)

fig.show()
fig.write_html("cilin_bert_3d.html")


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed